In [ ]:
import numpy as np
import torch as th
import matplotlib.pyplot as plt
import json
import subprocess

model = 'log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_rot2_dstL'
n_frames = 60
ckpt = 'ema_300000'
sample_folder = '/data/mint/sampling/TPAMI/main_result/ffhq/for_video_supp/'
sample_json = '/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/video_supp.json'
pf = '/valid/render_face/reverse_sampling/'

grid_size = 5
path = f'{sample_folder}/{model}/{ckpt}/{pf}/'

with open(sample_json, 'r') as f:
    sj = json.load(f)['pair']


vids = []
for pid, pair in sj.items():
    tmp = f'{path}/src={pair["src"]}/dst={pair["dst"]}/Lerp_1000/n_frames={n_frames}/'
    vid = f'{tmp}/res.mp4'
    vids.append(vid)
    
# Use ffmpeg to concatenate videos as a grid into one following the grid size
# Generate the FFmpeg command
input_cmds = []
for i, vid in enumerate(vids[:grid_size**2]):
    input_cmds.append(f"-i {vid}")

# Prepare xstack filter to form a 5x5 grid
layout = []
for row in range(grid_size):
    for col in range(grid_size):
        # Calculate the position based on `w` and `h` values
        # w0, w1, etc. represent the cumulative width up to the column index
        # h0, h1, etc. represent the cumulative height up to the row index
        x_pos = "+".join([f"w{c}" for c in range(col)]) if col > 0 else "0"
        y_pos = "+".join([f"h{r}" for r in range(row)]) if row > 0 else "0"
        layout.append(f"{x_pos}_{y_pos}")

xstack_filter = f"xstack=inputs={grid_size**2}:layout=" + "|".join(layout)
print(xstack_filter)


output_path = './grid1.mp4'
# Complete the FFmpeg command
ffmpeg_cmd = f"ffmpeg -y {' '.join(input_cmds)} -filter_complex \"{xstack_filter}\" -c:v libx264 -crf 23 -preset veryfast {output_path}"

# Run the command
subprocess.run(ffmpeg_cmd, shell=True)

xstack=inputs=4:layout=0_0|w0_0|0_h0|w0_h0


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

CompletedProcess(args='ffmpeg -y -i /data/mint/sampling/TPAMI/main_result/ffhq/for_video_supp//log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_rot2_dstL/ema_300000//valid/render_face/reverse_sampling///src=63295.jpg/dst=60065.jpg/Lerp_1000/n_frames=60//res.mp4 -i /data/mint/sampling/TPAMI/main_result/ffhq/for_video_supp//log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_rot2_dstL/ema_300000//valid/render_face/reverse_sampling///src=69545.jpg/dst=60065.jpg/Lerp_1000/n_frames=60//res.mp4 -i /data/mint/sampling/TPAMI/main_result/ffhq/for_video_supp//log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_rot2_dstL/ema_300000//valid/render_face/reverse_sampling///src=66943.jpg/dst=60065.jpg/Lerp_1000/n_frames=60//res.mp4 -i /data/mint/sampling/TPAMI/main_result/ffhq/for_video_supp//log=paired+difareli+cs+nodpm+trainset_256_cfg=paired+difareli+cs+nodpm+trainset_256.yaml_rot2_dstL